# ⚡ SoulX-FlashHead 1.3B Official GPU Avatar Server (Colab T4 GPU)

This notebook runs the official **SoulX-FlashHead 1.3B** (Real-Time Audio-Driven Digital Human) on Google Colab's free **NVIDIA T4 GPU (16 GB VRAM)**.

### Instructions:
1. Top menu: **Runtime > Change runtime type > T4 GPU > Save**.
2. Enter your free **Ngrok Auth Token** in Step 3.
3. Click **Runtime > Run all** (`Ctrl+F9`).
4. Copy the public **`https://xxxx-xx.ngrok-free.app/generate`** URL into your local `backend/.env`!

In [ ]:
# Step 1: Verify NVIDIA T4 GPU
!nvidia-smi

In [ ]:
# Step 2: Install All Dependencies & Download SoulX-FlashHead 1.3B Checkpoints
import os
from pathlib import Path

%cd /content
if not os.path.exists("/content/SoulX-FlashHead"):
    !git clone https://github.com/Soul-AILab/SoulX-FlashHead.git /content/SoulX-FlashHead

%cd /content/SoulX-FlashHead

# Install xfuser, pyloudnorm, loguru, and all diffusion transformer packages
!pip install --quiet fastapi uvicorn pyngrok python-multipart
!pip install --quiet pyloudnorm xfuser loguru einops omegaconf decord moviepy scipy av easydict insightface torchaudio torchvision diffusers transformers accelerate huggingface_hub librosa soundfile opencv-python-headless
!pip install --quiet -r requirements.txt 2>/dev/null || true

# Download SoulX-FlashHead 1.3B Checkpoints from Hugging Face
checkpoints_dir = Path("/content/SoulX-FlashHead/checkpoints")
if not checkpoints_dir.exists() or len(list(checkpoints_dir.glob("*"))) == 0:
    print("⏳ Downloading SoulX-FlashHead 1.3B checkpoints...")
    !python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='Soul-AILab/SoulX-FlashHead-1_3B', local_dir='/content/SoulX-FlashHead/checkpoints', local_dir_use_symlinks=False)"
else:
    print("⚡ Checkpoints already downloaded, skipping download!")

print("✅ SoulX-FlashHead 1.3B Environment Ready!")

In [ ]:
# Step 3: Create FastAPI Microservice for SoulX-FlashHead
server_code = '''
import os
import glob
import time
import shutil
import subprocess
from pathlib import Path
import uvicorn
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="SoulX-FlashHead 1.3B GPU Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

BASE_DIR = Path("/content/SoulX-FlashHead")
TEMP_DIR = Path("/content/avatar_jobs")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

@app.get("/")
def status():
    return {"status": "online", "engine": "SoulX-FlashHead 1.3B Diffusion", "device": "NVIDIA T4 GPU (16GB VRAM)"}

@app.post("/generate")
async def generate_avatar(photo: UploadFile = File(...), audio: UploadFile = File(...)):
    job_id = f"job_{int(time.time() * 1000)}"
    job_dir = TEMP_DIR / job_id
    job_dir.mkdir(parents=True, exist_ok=True)

    img_path = job_dir / "source_photo.jpg"
    wav_path = job_dir / "source_audio.wav"
    out_dir = job_dir / "results"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "avatar.mp4"

    with open(img_path, "wb") as f: f.write(await photo.read())
    with open(wav_path, "wb") as f: f.write(await audio.read())

    start_t = time.time()
    print(f"[SOULX-FLASHHEAD] Generating real-time digital human video on T4 GPU for {job_id}...")

    # Set PYTHONPATH to include SoulX-FlashHead root
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{str(BASE_DIR)}:{env.get('PYTHONPATH', '')}"

    # Check for flash_head/inference.py or generate_video.py
    if (BASE_DIR / "flash_head" / "inference.py").exists():
        cmd = [
            "python", "-m", "flash_head.inference",
            "--image_path", str(img_path),
            "--audio_path", str(wav_path),
            "--output_path", str(out_path),
            "--checkpoint_dir", str(BASE_DIR / "checkpoints")
        ]
    else:
        cmd = [
            "python", "generate_video.py",
            "--image_path", str(img_path),
            "--audio_path", str(wav_path),
            "--output_path", str(out_path),
            "--checkpoint_dir", str(BASE_DIR / "checkpoints")
        ]

    res = subprocess.run(cmd, cwd=str(BASE_DIR), env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # Locate rendered MP4
    mp4_files = sorted(list(job_dir.glob("*.mp4")) + list(job_dir.glob("**/*.mp4")), key=os.path.getmtime, reverse=True)
    if not mp4_files or mp4_files[0].stat().st_size < 1000:
        err_output = res.stderr.decode("utf-8", errors="ignore")[-700:]
        print(f"[ERROR] SoulX-FlashHead failed: {err_output}")
        raise HTTPException(status_code=500, detail=f"SoulX-FlashHead inference failed: {err_output}")

    rendered_file = mp4_files[0]
    elapsed = time.time() - start_t
    print(f"[SOULX-FLASHHEAD] Rendered successfully in {elapsed:.2f}s! ({rendered_file.stat().st_size} bytes)")

    return FileResponse(path=str(rendered_file), media_type="video/mp4", filename="avatar.mp4")
'''

with open("/content/SoulX-FlashHead/server.py", "w") as f:
    f.write(server_code)
print("✅ SoulX-FlashHead 1.3B server written!")

In [ ]:
# Step 4: Open Ngrok Tunnel & Start SoulX-FlashHead Server
from pyngrok import ngrok

# Enter your ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ""  # <-- Paste your ngrok auth token here
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

ngrok.kill()
public_url = ngrok.connect(8000).public_url
print("=" * 75)
print("⚡ YOUR OFFICIAL SOULX-FLASHHEAD 1.3B SERVER IS READY!")
print(f"Endpoint: {public_url}/generate")
print("=" * 75)
print("Update your local backend/.env with:")
print("AVATAR_PROVIDER=gpu")
print(f"AVATAR_GPU_URL={public_url}/generate")
print("=" * 75)

# Launch server on GPU
!python -m uvicorn server:app --app-dir /content/SoulX-FlashHead --host 0.0.0.0 --port 8000